# Reproduce the UAV wildlife training workflow

Run the cells in order. This notebook loads the prepared Thermal and RGB ZIPs from the project's Google Drive links. Each user runs the notebook in their own Colab account and runtime.

中文：按顺序运行各单元格，自动下载、解压并检查数据。训练前选择 GPU 并开启 RUN_TRAINING。若下载失败，使用 mount 模式读取自己 Drive 中的数据快捷方式。


In [ ]:
%pip install -q "ultralytics>=8.3,<9" "gdown>=5,<6" "PyYAML>=6"

from collections import Counter
from pathlib import Path
import torch
from ultralytics import YOLO

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Select Runtime > Change runtime type > GPU before training.')

## 1. Load the prepared datasets from Google Drive

Default download mode reads the two link-accessible project ZIPs (about 2.5 GB). For quota errors or private replacements, set DATA_SOURCE to 'mount', add ZIP shortcuts inside a folder in your own My Drive, and set DRIVE_DATA_DIR. Mounting authorizes your own Google account; it does not mount the repository author's account.

The ZIPs contain multiple historical datasets. Defaults select datasets/YOLO_Thermal (199 train / 50 validation images) and datasets/YOLO_RGB (216 / 55), which have no overlapping train/validation names in the checked copies. Larger archived versions have overlapping names. Versions are never combined, and class mappings are preserved. Set DATASET_SUBDIRS entries to None to list choices if an archive has multiple versions. This small selection supports workflow reproduction, not reproduction of the historical report metrics.

Original files are preserved. Old label caches are skipped and rebuilt by training. The RGB source imagery includes overlaid recording time and location. Dataset attribution: BAMBI, Praschl et al., DOI 10.5281/zenodo.18692354, CC BY 4.0.


In [ ]:
RUN_TRAINING = False  # Set True when ready to start the two 100-epoch runs.
DATA_SOURCE = 'download'  # 'download' or 'mount'
DRIVE_DATA_DIR = '/content/drive/MyDrive/UAV_Wildlife'  # used only by 'mount'
DATA_ROOT = Path('/content/datasets')
ARCHIVE_ROOT = Path('/content/dataset_archives')
DATASETS = {
    'YOLO_Thermal': '13b-M9OGh9kPc3DuzyZWQoSeAlBxCDoLe',
    'YOLO_RGB': '1mafY9A2mGX-xAcELYpK0zyg7lQEP6Bea',
}

# Optional: save training results directly to your Drive instead of runtime storage.
# Set this to True to mount Drive and preserve checkpoints if Colab disconnects.
SAVE_RESULTS_TO_DRIVE = False
RESULTS_ROOT = Path('/content/runs/detect')

# The linked ZIPs contain multiple versions. Select each dataset explicitly.
# 'datasets/YOLO_*' is the small version without overlapping train/val names.
# Use None to list choices and stop before extracting an ambiguous archive.
DATASET_SUBDIRS = {
    'YOLO_Thermal': 'datasets/YOLO_Thermal',
    'YOLO_RGB': 'datasets/YOLO_RGB',
}


In [ ]:
import json
import shutil
import stat
import zipfile
import yaml

def load_archive(name, file_id, source, archive_root, drive_dir=None):
    if source not in ('download', 'mount'):
        raise ValueError("DATA_SOURCE must be 'download' or 'mount'.")
    # Namespace by file ID so changing links cannot silently reuse another dataset.
    target = Path(archive_root) / file_id / f'{name}.zip'
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and zipfile.is_zipfile(target):
        print(f'Reusing {target}')
        return target
    partial = target.with_suffix('.zip.part')
    if source == 'mount':
        original = Path(drive_dir) / f'{name}.zip'
        if not original.is_file():
            raise FileNotFoundError(
                f'Missing {original}. Add a Drive shortcut to the ZIP in this folder, '
                'or correct DRIVE_DATA_DIR.'
            )
        shutil.copyfile(original, partial)
    else:
        import gdown
        try:
            result = gdown.download(id=file_id, output=str(partial), quiet=False, resume=True)
            if not result:
                raise RuntimeError('Drive did not return a file.')
        except Exception as exc:
            raise RuntimeError(
                f'Could not download {name}.zip. Check access to the link or switch '
                "DATA_SOURCE to 'mount' and set DRIVE_DATA_DIR; no sharing change is needed."
            ) from exc
    if not zipfile.is_zipfile(partial):
        partial.unlink(missing_ok=True)
        raise ValueError(f'{name}: download is not a ZIP (possibly a Drive permission page).')
    partial.replace(target)
    return target

def prepare_dataset(archive, name, destination, dataset_subdir=None):
    archive = Path(archive)
    destination = Path(destination).resolve()
    # Inspect ZIP members before extracting; some project archives contain several versions.
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError(f'Unsafe archive entry: {member.filename}')
        members = z.namelist()
        candidates = []
        for yaml_name in members:
            if Path(yaml_name).name != 'data.yaml' or '__MACOSX' in Path(yaml_name).parts:
                continue
            prefix = yaml_name[:-len('data.yaml')]
            if all(any(n.startswith(prefix + f'{group}/{split}/') and not n.endswith('/')
                       for n in members) for group in ('images', 'labels') for split in ('train', 'val')):
                candidates.append(prefix)
        if not candidates:
            raise ValueError(f'{name}: no annotated YOLO dataset found; raw videos cannot be used directly.')
        print(f'{name}: available dataset directories:')
        for prefix in candidates:
            info = yaml.safe_load(z.read(prefix + 'data.yaml'))
            counts = {s: sum(n.startswith(prefix + f'images/{s}/') and Path(n).suffix.lower()
                            in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
                            for n in members) for s in ('train', 'val')}
            print(' ', prefix.rstrip('/') or '.', counts, 'classes:', info.get('names'))
        if dataset_subdir is None:
            if len(candidates) != 1:
                raise ValueError(f'{name}: multiple dataset versions. Set DATASET_SUBDIRS[{name!r}] '
                                 'to one of the printed directories and rerun. Do not mix versions.')
            prefix = candidates[0]
        else:
            prefix = str(dataset_subdir).strip('/')
            prefix = '' if prefix == '.' else prefix + '/'
            if prefix not in candidates:
                raise ValueError(f'{name}: selected directory {dataset_subdir!r} is not a valid dataset.')
        destination.mkdir(parents=True, exist_ok=True)
        marker = destination / '.extracted.json'
        signature = {'archive': str(archive.resolve()), 'size': archive.stat().st_size,
                     'mtime_ns': archive.stat().st_mtime_ns, 'dataset_subdir': prefix}
        previous = json.loads(marker.read_text()) if marker.is_file() else None
        if previous != signature:
            if any(destination.iterdir()):
                raise RuntimeError(f'{destination} has an incomplete or different extraction. '
                                   'Choose a fresh DATA_ROOT and rerun the load cell.')
            # Extract only this version's YAML, images and labels, not other archived copies.
            selected = [m for m in z.infolist() if m.filename == prefix + 'data.yaml'
                        or m.filename.startswith(prefix + 'images/')
                        or m.filename.startswith(prefix + 'labels/')]
            selected = [m for m in selected if not m.filename.endswith('.cache')]
            for member in selected:
                target = (destination / member.filename).resolve()
                if not target.is_relative_to(destination) or stat.S_ISLNK(member.external_attr >> 16):
                    raise ValueError(f'Unsafe archive entry: {member.filename}')
            z.extractall(destination, members=selected)
    root = destination / prefix
    yaml_path = root / 'data.yaml'
    config = yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
    if not isinstance(config, dict) or not config.get('names'):
        raise ValueError(f'Missing class names in {yaml_path}')
    # Normalize stale Mac/Windows paths in the local copy only.
    config.update(path=str(root), train='images/train', val='images/val')
    config.pop('download', None)
    if 'test' in config:
        if (root / 'images/test').is_dir():
            config['test'] = 'images/test'
        else:
            config.pop('test')
    yaml_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    marker.write_text(json.dumps(signature), encoding='utf-8')
    print(f'{name}: ready at {root}')
    return root


In [ ]:
DATASETS_VALIDATED = False
if DATA_SOURCE == 'mount' or SAVE_RESULTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if SAVE_RESULTS_TO_DRIVE:
    RESULTS_ROOT = Path('/content/drive/MyDrive/UAV_Wildlife_training/runs/detect')

DATASET_ROOTS = {}
for name, file_id in DATASETS.items():
    archive = load_archive(name, file_id, DATA_SOURCE, ARCHIVE_ROOT, DRIVE_DATA_DIR)
    # Each file gets its own extraction directory; nested ZIP layouts also work.
    DATASET_ROOTS[name] = prepare_dataset(archive, name, DATA_ROOT / file_id, DATASET_SUBDIRS.get(name))
print('Training results will be saved to:', RESULTS_ROOT)

## 2. Inspect the datasets before training

Empty .txt labels are valid negative frames. Stop on empty splits, missing labels, orphan labels, overlapping train/validation names, or invalid YOLO annotations. Class names come from each dataset's own YAML.


In [ ]:
DATASETS_VALIDATED = False
import math

if set(DATASET_ROOTS) != set(DATASETS):
    raise RuntimeError('Load all selected datasets successfully before inspection.')

for modality, root in DATASET_ROOTS.items():
    config = yaml.safe_load((root / 'data.yaml').read_text())
    names = config['names']
    class_names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    print(f'\n{modality}')
    split_stems = {}
    for split in ('train', 'val'):
        images = [p for p in (root / 'images' / split).rglob('*')
                  if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}]
        labels = list((root / 'labels' / split).rglob('*.txt'))
        image_keys = {p.relative_to(root / 'images' / split).with_suffix('') for p in images}
        label_keys = {p.relative_to(root / 'labels' / split).with_suffix('') for p in labels}
        if not images or image_keys != label_keys or len(image_keys) != len(images):
            raise ValueError(f'{modality}/{split}: empty split, duplicate image stems or unmatched images/labels. '
                             f'Missing labels: {len(image_keys - label_keys)}; orphan labels: {len(label_keys - image_keys)}')
        split_stems[split] = image_keys
        counts = Counter()
        negatives = 0
        for label in labels:
            lines = [line for line in label.read_text().splitlines() if line.strip()]
            negatives += not lines
            for number, line in enumerate(lines, 1):
                parts = line.split()
                if len(parts) != 5:
                    raise ValueError(f'{label}:{number}: expected 5 YOLO values')
                cid = int(parts[0])
                x, y, w, h = map(float, parts[1:])
                if (cid not in class_names or not all(math.isfinite(v) for v in (x, y, w, h))
                        or not 0 <= x <= 1 or not 0 <= y <= 1 or not 0 < w <= 1 or not 0 < h <= 1):
                    raise ValueError(f'{label}:{number}: invalid class or normalized box')
                counts[cid] += 1
        print(split, {'images': len(images), 'labels': len(labels), 'negative_frames': negatives})
        print({class_names[key]: value for key, value in sorted(counts.items())})
    if split_stems['train'] & split_stems['val']:
        raise ValueError(f'{modality}: train/val contain overlapping image names')
DATASETS_VALIDATED = True


## 3. Train the documented configurations

Select a GPU runtime and set RUN_TRAINING = True in the configuration cell when ready, then rerun configuration and this training cell. Results are stored under RESULTS_ROOT; use SAVE_RESULTS_TO_DRIVE = True and rerun the loading cell to persist them in your own Drive. Local /content storage disappears when the runtime is deleted. Reduce BATCH_SIZE if RGB at 1280 px runs out of GPU memory, and record the changed configuration. The initial yolov8n.pt is generic pretrained initialization, not the project-trained checkpoint.


In [ ]:
if not RUN_TRAINING:
    print('Data is ready. Set RUN_TRAINING = True in section 1, rerun that config cell, then this training cell.')
else:
    if not globals().get('DATASETS_VALIDATED', False):
        raise RuntimeError('Run the dataset inspection successfully before training.')
    if not torch.cuda.is_available():
        raise RuntimeError('Select a GPU runtime, then rerun setup before training.')
    
    BATCH_SIZE = 16
    FINAL_RUNS = {
        'thermal': {'data': DATASET_ROOTS['YOLO_Thermal'] / 'data.yaml', 'imgsz': 640},
        'rgb': {'data': DATASET_ROOTS['YOLO_RGB'] / 'data.yaml', 'imgsz': 1280},
    }
    
    for modality, config in FINAL_RUNS.items():
        model = YOLO('yolov8n.pt')
        model.train(
            data=str(config['data']),
            epochs=100,
            imgsz=config['imgsz'],
            batch=BATCH_SIZE,
            optimizer='SGD',
            patience=20,
            name=f'{modality}_yolov8n_{config["imgsz"]}px_sgd',
            project=str(RESULTS_ROOT),
            save=True,
            plots=True,
        )


## 4. Reported experiment evidence

Original saved runs reported Thermal mAP@50 = **84.70%** and RGB mAP@50 = **85.93%**. These are historical results; the selected smaller datasets, dependency versions and randomness can produce different metrics. Exact reproduction requires the original evaluation split, annotations, checkpoints and software settings. The calculation below recomputes the published OOD count ratio only, not object-level recall.


In [ ]:
detections, annotations = 555, 572
coverage = detections / annotations
print(f'OOD count coverage: {coverage:.2%}')
print('Do not interpret this as object-level recall without one-to-one matching.')